# 04 — Statistical testing (Mann–Whitney U)

**Section:** Evaluation layer
**Depends on:** `03_promethee_ranking.ipynb` (`artifacts/reports/promethee_ranked_rules.csv`), `02b_extraction_evolution.ipynb` (`per_run_best_fidelity.csv`)
**Feeds into:** the results write-up

## Design

**Primary test (methodology §4.6.2, locked):** two-sided Mann–Whitney U on the unified φ values, α = 0.05, for
1. baseline vs SAM (pools GA and NSGA-II within each condition), and
2. GA vs NSGA-II (pools baseline and SAM within each extractor).

**Why it needs a companion analysis.** The pooled rule sets are small (10 vs 9 and 13 vs 6 rules). Rules within a pool are not independent draws: they are the survivors of a Pareto filter, many come from the same run, and φ values come from one pairwise-comparison run, so they are mutually dependent (they sum to zero). The test is run as specified, but its power is very low, and it is read together with:
- the same test on each raw criterion (to see what drives any difference),
- an exact permutation test (to confirm the p-values without the normal approximation), and
- a **supplementary per-run analysis**: the 30 extraction runs per (extractor, condition) are genuine independent replicates, so comparing the best fidelity each run achieved gives a far better-powered test of the same question.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")            # run from the repo root so relative paths match the pipeline
sys.path.insert(0, ".")
import numpy as np, pandas as pd

from scipy.stats import mannwhitneyu, wilcoxon, permutation_test
from src.evaluation import stage5_utils as U
REPORT = U.REPORT_DIR
ranked = pd.read_csv(REPORT / "promethee_ranked_rules.csv")
print(f"loaded {len(ranked)} ranked rules"); print(ranked.groupby(["extractor", "condition"]).size())

def mwu(a, b, la, lb, alpha=0.05):
    a, b = np.asarray(a, float), np.asarray(b, float)
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    r = 1 - 2 * u / (len(a) * len(b))        # rank-biserial; sign follows the original notebook (negative: A tends to be higher)
    return {"comparison": f"{la} vs {lb}", "n_a": len(a), "n_b": len(b), "median_a": float(np.median(a)),
            "median_b": float(np.median(b)), "U": float(u), "p_value": float(p), "significant_at_0.05": bool(p < alpha),
            "rank_biserial": float(r)}

loaded 19 ranked rules
extractor  condition
ga         baseline     7
           sam          6
nsga       baseline     3
           sam          3
dtype: int64


## 1. Primary test on φ

In [2]:
groups = {
    "baseline vs sam": (ranked[ranked.condition == "baseline"], ranked[ranked.condition == "sam"]),
    "ga vs nsga":      (ranked[ranked.extractor == "ga"],       ranked[ranked.extractor == "nsga"]),
}
primary = pd.DataFrame([mwu(a.phi, b.phi, *k.split(" vs ")) for k, (a, b) in groups.items()])
primary

,comparison,n_a,n_b,median_a,median_b,U,p_value,significant_at_0.05,rank_biserial
0,baseline vs sam,10,9,0.041787,0.089230,42.0,0.838256,False,0.066667
1,ga vs nsga,13,6,0.129561,-0.258822,76.0,0.000295,True,-0.948718


## 2. What drives it: the same test on each criterion, plus an exact permutation check
`rank_biserial` is negative when group A tends to have the **higher** value of that column.

In [3]:
rows = []
for k, (a, b) in groups.items():
    for col in ["phi", "fidelity_test", "stability", "length", "coverage_test"]:
        r = mwu(a[col], b[col], *k.split(" vs "))
        perm = permutation_test((a[col].to_numpy(float), b[col].to_numpy(float)),
                                lambda x, y: np.median(x) - np.median(y), n_resamples=20000, random_state=0,
                                alternative="two-sided").pvalue
        rows.append({"groups": k, "variable": col, "median_a": r["median_a"], "median_b": r["median_b"],
                     "U": r["U"], "p_mwu": r["p_value"], "p_permutation_(diff of medians)": float(perm), "rank_biserial": r["rank_biserial"]})
per_criterion = pd.DataFrame(rows)
per_criterion.round(4)

,groups,variable,median_a,median_b,U,p_mwu,p_permutation_(diff of medians),rank_biserial
0,baseline vs sam,phi,0.0418,0.0892,42.0,0.8383,0.8827,0.0667
1,baseline vs sam,fidelity_test,0.4706,0.4794,36.0,0.4877,0.5162,0.2000
2,baseline vs sam,stability,0.0192,0.0124,43.0,0.9025,0.8620,0.0444
3,baseline vs sam,length,2.0000,2.0000,39.5,0.6720,1.0000,0.1222
4,baseline vs sam,coverage_test,0.0276,0.0315,43.5,0.9349,0.6418,0.0333
5,ga vs nsga,phi,0.1296,-0.2588,76.0,0.0003,0.0025,-0.9487
6,ga vs nsga,fidelity_test,0.4788,0.4547,77.0,0.0001,0.0027,-0.9744
7,ga vs nsga,stability,0.0057,0.0539,4.0,0.0009,0.0022,0.8974
8,ga vs nsga,length,2.0000,2.0000,47.5,0.4668,1.0000,-0.2179
9,ga vs nsga,coverage_test,0.0252,0.0424,31.0,0.5105,0.2842,0.2051


## 3. How much can this sample detect?

Simulated power of the two-sided Mann–Whitney test at α = 0.05 for a shift of d standard deviations between two normal groups of the observed sizes.

In [4]:
rng = np.random.default_rng(1)
def power(n1, n2, d, sims=3000):
    hit = 0
    for _ in range(sims):
        hit += mannwhitneyu(rng.normal(0, 1, n1), rng.normal(d, 1, n2), alternative="two-sided").pvalue < .05
    return hit / sims
pw = pd.DataFrame({"shift d (SD)": [0.5, 1.0, 1.5, 2.0],
                   "power, 10 vs 9 (baseline vs SAM)": [power(10, 9, d) for d in (0.5, 1, 1.5, 2)],
                   "power, 13 vs 6 (GA vs NSGA-II)":   [power(13, 6, d) for d in (0.5, 1, 1.5, 2)],
                   "power, 30 vs 30 (per-run, §4)":     [power(30, 30, d) for d in (0.5, 1, 1.5, 2)]})
pw.round(2)

,shift d (SD),"power, 10 vs 9 (baseline vs SAM)","power, 13 vs 6 (GA vs NSGA-II)","power, 30 vs 30 (per-run, §4)"
0,0.5,0.15,0.14,0.46
1,1.0,0.48,0.44,0.96
2,1.5,0.83,0.78,1.00
3,2.0,0.98,0.95,1.00


## 4. Supplementary: per-run comparison (30 independent runs per group)

Statistic per run: the best fidelity (train) in the run's final front. Runs are independent replicates. For baseline vs SAM the same seeds were used in both conditions, so the 30 runs are also **paired by seed** (identical starting populations); both the independent test and the paired Wilcoxon signed-rank test are shown. GA vs NSGA-II uses the independent test only.

In [5]:
pr = pd.read_csv(REPORT / "per_run_best_fidelity.csv")
rows = []
for e in ("ga", "nsga"):
    a = pr[(pr.extractor == e) & (pr.condition == "baseline")].sort_values("run_id")
    b = pr[(pr.extractor == e) & (pr.condition == "sam")].sort_values("run_id")
    r = mwu(a.best_fidelity_train, b.best_fidelity_train, f"{e}: baseline", f"{e}: sam")
    w = wilcoxon(a.best_fidelity_train.to_numpy(), b.best_fidelity_train.to_numpy(), zero_method="wilcox")
    r.update({"paired_wilcoxon_p": float(w.pvalue), "runs_where_sam_higher": int((b.best_fidelity_train.to_numpy() > a.best_fidelity_train.to_numpy()).sum()),
              "mean_diff_sam_minus_baseline": float((b.best_fidelity_train.to_numpy() - a.best_fidelity_train.to_numpy()).mean())})
    rows.append(r)
for c in ("baseline", "sam"):
    a = pr[(pr.extractor == "ga") & (pr.condition == c)]; b = pr[(pr.extractor == "nsga") & (pr.condition == c)]
    rows.append(mwu(a.best_fidelity_train, b.best_fidelity_train, f"{c}: ga", f"{c}: nsga"))
per_run = pd.DataFrame(rows)
per_run.round(5)

,comparison,n_a,n_b,median_a,median_b,U,p_value,significant_at_0.05,rank_biserial,paired_wilcoxon_p,runs_where_sam_higher,mean_diff_sam_minus_baseline
0,ga: baseline vs ga: sam,30,30,0.46906,0.47044,369.0,0.23399,False,0.18000,0.30852,19.0,0.00291
1,nsga: baseline vs nsga: sam,30,30,0.40664,0.40673,480.0,0.66273,False,-0.06667,0.24495,11.0,-0.00270
2,baseline: ga vs baseline: nsga,30,30,0.46906,0.40664,895.0,0.00000,True,-0.98889,NaN,NaN,NaN
3,sam: ga vs sam: nsga,30,30,0.47044,0.40673,889.0,0.00000,True,-0.97556,NaN,NaN,NaN


## 5. Save

In [6]:
primary.to_csv(REPORT / "mannwhitney_results.csv", index=False)
per_criterion.to_csv(REPORT / "mannwhitney_per_criterion.csv", index=False)
per_run.to_csv(REPORT / "mannwhitney_per_run_supplementary.csv", index=False)
pw.to_csv(REPORT / "mannwhitney_power.csv", index=False)
print("saved to", REPORT)

saved to artifacts/reports


## Interpretation notes (fill from the printed results)

- State the primary result exactly as computed, with the effect size and the sample sizes, and say that the test has little power at these n (§3).
- Use §2 to say which criterion carries any difference, and §4 to say whether a larger independent sample (runs) supports or contradicts the primary result.
- Do not describe a non-significant result as evidence of "no effect"; it is absence of evidence at this sample size.